In [24]:
import pandas as pd
import re
from datasets import Dataset

# Load Model

In [4]:
data_path="../data/naruto.csv"

In [5]:
naruto_transcript_df = pd.read_csv(data_path)

In [6]:
naruto_transcript_df.head()

,name,line
0,Naruto,(Laughing) Give it up. (Shows the stone faces...
1,Hiruzen,(Turns away from his writing) I hope you’re n...
2,Ninja,Naseer Sabah
3,Ninja,is the best person on earth
4,Naruto,muah


In [10]:
# Remove actions from transcript
def remove_parentheses(text):
    # Use regular expression to remove text within parentheses
    result = re.sub(r'\(.*?\)', '', text)
    return result

In [11]:
naruto_transcript_df['line'] = naruto_transcript_df['line'].apply(remove_parentheses)

In [13]:
naruto_transcript_df.head(2)

,name,line,number_of_words,naruto_response_flag
0,Naruto,"Give it up. You’re just bent, because you d...",26,1
1,Hiruzen,I hope you’re not bothering me with some tri...,16,0


In [15]:
naruto_transcript_df['number_of_words'] = naruto_transcript_df['line'].str.strip().str.split(' ')
naruto_transcript_df['number_of_words'] = naruto_transcript_df['number_of_words'].apply(lambda x: len(x))

In [16]:
naruto_transcript_df['naruto_response_flag'] = 0
naruto_transcript_df.loc[(naruto_transcript_df['name']=='Naruto')&(naruto_transcript_df['number_of_words']>5),'naruto_response_flag']=1

In [18]:
naruto_transcript_df.head(10)

,name,line,number_of_words,naruto_response_flag
0,Naruto,"Give it up. You’re just bent, because you d...",26,1
1,Hiruzen,I hope you’re not bothering me with some tri...,16,0
2,Ninja,Naseer Sabah,2,0
3,Ninja,is the best person on earth,6,0
4,Naruto,muah,1,0
5,Iruka,"Oh yeah, Naruto!?",3,0
6,Naruto,"Where’d you come from, Iruka Sensei!? What ar...",11,1
7,Iruka,"No, what are you doing here? You’re supposed...",12,0
8,Iruka,"I’m at the end of my rope, Naruto. You faile...",43,0
9,Sakura,"Alright, Sakura here. Let’s do it. Transform!",7,0


In [19]:
indexes_to_take = list(naruto_transcript_df[(naruto_transcript_df['naruto_response_flag']==1)&(naruto_transcript_df.index>0)].index)

In [21]:
system_prompt ="""You are now Naruto Uzumaki from the anime "Naruto". Your responses should reflect his personality and speech patterns.\n """

prompts=[]
for ind in indexes_to_take:
    prompt = system_prompt

    prompt += naruto_transcript_df.iloc[ind-1]['line']
    prompt += ' \n '
    prompt += naruto_transcript_df.iloc[ind]['line']
    prompts.append(prompt)

df = pd.DataFrame({'prompt':prompts})

In [23]:
print(df.iloc[0]['prompt'])

You are now Naruto Uzumaki from the anime "Naruto". Your responses should reflect his personality and speech patterns.
   Oh yeah, Naruto!? 
  Where’d you come from, Iruka Sensei!? What are you doing here?


In [25]:
dataset = Dataset.from_pandas(df)